In [1]:
import pandas as pd
from sgp4.api import Satrec
from skyfield.api import EarthSatellite, load
import datetime

In [ ]:
df = pd.read_json('44204_2024yr.json')
selected_data = df[['EPOCH', 'TLE_LINE0', 'TLE_LINE1', 'TLE_LINE2']]

selected_data.to_csv('sgp4_C38_data_2024.csv', index=False, encoding='utf-8')

In [4]:
df = pd.read_csv('PC38_data_2024_sp3.csv')
# df2 = pd.read_csv('sgp4_C06_data.csv')

df['epoch'] = pd.to_datetime(df['epoch'])

df_selected = df[(df['epoch'].dt.hour == 12) & (df['epoch'].dt.minute == 0) & (df['epoch'].dt.second == 0)]
df_selected.to_csv('sp3_C38_2024_onePerDay.csv', index=False, encoding='utf-8')

In [ ]:
df = pd.read_csv('sgp4_C38_data_2024.csv')

df['EPOCH'] = pd.to_datetime(df['EPOCH'])
df_one_per_day = df.groupby(df['EPOCH'].dt.date).first() 
df_one_per_day.to_csv('sgp4_C38_2024_onePerDay.csv', index=False, encoding='utf-8')

In [ ]:
# # Calculating TLE into TEME

# INPUT_CSV = 'tle_C06_2025_onePerDay.csv'
# OUTPUT_CSV = 'sgp4_TEME_C06_2025_4perRecord.csv'
# L1 = 'TLE_LINE1'
# L2 = 'TLE_LINE2'

# df = pd.read_csv(INPUT_CSV)
# w = []

# day_offset = [0, 1, 3, 5]

# for index, row in df.iterrows():
#     l1 = row[L1]
#     l2 = row[L2]

#     record = row.to_dict()

#     try:
#         # Inicjalizacja modelu SGP4 dla danego TLE
#         sat = Satrec.twoline2rv(l1, l2)
        
#         jd_epoka = sat.jdsatepoch
#         fr_epoka = sat.jdsatepochF
        
#         # Obliczenia dla każdego przesunięcia w dniach
#         for offset in day_offset:
#             jd_cel = jd_epoka + offset
            
#             e, r, v = sat.sgp4(jd_cel, fr_epoka)
            
#             if e == 0:
#                 record[f'X_{offset}d [km]'] = r[0]
#                 record[f'Y_{offset}d [km]'] = r[1]
#                 record[f'Z_{offset}d [km]'] = r[2]
#                 record[f'Vx_{offset}d [km/s]'] = v[0]
#                 record[f'Vy_{offset}d [km/s]'] = v[1]
#                 record[f'Vz_{offset}d [km/s]'] = v[2]
#             else:
#                 # Błąd 
#                 record[f'X_{offset}d [km]'] = None
#                 record[f'Y_{offset}d [km]'] = None
#                 record[f'Z_{offset}d [km]'] = None
#                 record[f'Vx_{offset}d [km/s]'] = None
#                 record[f'Vy_{offset}d [km/s]'] = None
#                 record[f'Vz_{offset}d [km/s]'] = None
                
#     except Exception as ex:
#         print(f"Błąd przetwarzania wiersza {index}: {ex}")
        
#     w.append(record)

# df_wyniki = pd.DataFrame(w)
# df_wyniki.to_csv(OUTPUT_CSV, index=False)

# print(f"Zakończono! Zapisano wyniki do {OUTPUT_CSV}")


Zakończono! Zapisano wyniki do sgp4_TEME_C06_2025_4perRecord.csv


In [ ]:
import pandas as pd
import datetime
from skyfield.api import EarthSatellite, load
from skyfield.framelib import itrs

INPUT_CSV = 'sgp4_C38_2024_onePerDay.csv'
OUTPUT_CSV = 'ECEF_C38_2024_4perRecord.csv'

SAT_ID = 'PC38' 

ts = load.timescale()
df = pd.read_csv(INPUT_CSV) 

w = []
day_offset = [0, 1, 3, 5]

for index, row in df.iterrows():
    l1 = row['TLE_LINE1'] 
    l2 = row['TLE_LINE2'] 
    
    record = row.to_dict()
    record['sat_id'] = SAT_ID
    
    try:
        sat = EarthSatellite(l1, l2, SAT_ID, ts)
        
        dt_epoka = sat.epoch.utc_datetime()
        
        for offset in day_offset:
            dt_target = (dt_epoka + datetime.timedelta(days=offset)).replace(hour=12, minute=0, second=0, microsecond=0)
            
            t_cel = ts.from_datetime(dt_target)
            
            geocentric = sat.at(t_cel)
            pos_ecef = geocentric.frame_xyz(itrs).km
            
            record[f'X_{offset}d [km]'] = round(pos_ecef[0], 6)
            record[f'Y_{offset}d [km]'] = round(pos_ecef[1], 6)
            record[f'Z_{offset}d [km]'] = round(pos_ecef[2], 6)
            record[f'Epoch_{offset}d'] = dt_target.strftime('%Y-%m-%d %H:%M:%S')
            
    except Exception as ex:
        print(f"Błąd wiersza {index}: {ex}")
        for offset in day_offset:
            record[f'X_{offset}d [km]'] = None
            record[f'Y_{offset}d [km]'] = None
            record[f'Z_{offset}d [km]'] = None
            record[f'Epoch_{offset}d'] = None
            
    w.append(record)

df_wyniki = pd.DataFrame(w)
df_wyniki.to_csv(OUTPUT_CSV, index=False)

print(f"Zapisano wyniki: {OUTPUT_CSV}")

FileNotFoundError: [Errno 2] No such file or directory: 'sgp4_C38_2024_onePerDay.csv'